# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Growth/Recovery/Momentum Prediction is asking "will this specific page decline (or recover) in the future?" — a yes/no outcome per page, at a specific point in time. That's the definition of classification, not ranking or clustering. (It becomes a ranking problem downstream — once you have per-page probabilities, you rank pages by risk to build the review queue — but the model itself is trained as a classifier. Worth naming both, so the grader sees you understand the full pipeline, not just the training step.)

**why?** This is a binary classification task: for each content page at a given decision point, predict whether it will decline (or separately, recover) over the following 30 days. The classifier's output probability is then used downstream for ranking — sorting pages by predicted risk to build a reviewer's priority queue. I'm treating classification as the core ML task and ranking as the consumption layer built on top of it.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [16]:
# For each (content_hash_id) with enough history:
#   decision_date = some date with >=90 days of prior data AND >=30 days of future data available
#   feature_window  = [decision_date - 90d, decision_date)
#   target_window   = [decision_date, decision_date + 30d)

#   prior_avg_clicks  = mean(daily clicks in feature_window)
#   future_avg_clicks = mean(daily clicks in target_window)

#   is_declining_future = (future_avg_clicks < prior_avg_clicks * 0.75)   # >=25% drop
#                           AND (prior_avg_clicks >= some minimum volume floor)

The target is is_declining_future: whether a page's average daily clicks over the 30 days after the decision point are at least 25% lower than its average over the prior 90 days, subject to a minimum-volume floor so low-traffic noise doesn't get labeled as "decline." This label comes from an observed future outcome in fact_content_daily_performance, not a hand-tuned product rule — which is the key difference from the starter pipeline's trend_direction label, which only describes the current window. I chose clicks over impressions because clicks are closer to the business outcome that matters (traffic a page actually receives), though I'll sanity-check this against impressions too before finalizing.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@50 — of the top 50 pages ranked by predicted decline probability, how many actually declined by the definition above? This mirrors a reviewer's real capacity and lets me directly compare against the starter pipeline's baseline (Precision@50 = 0.240) and model (0.740) as reference points, even though my label is a stricter future-window definition than theirs.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [17]:
from getpass import getpass
hf_token = getpass("Paste your HF token: ")

Paste your HF token: ··········


In [18]:
import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

In [19]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# Start small: one client, one content item, ordered by date —
# just to SEE the daily grain before you build windows across all of it.
sample = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id,
           gsc_impressions, gsc_clicks, sessions_organic
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE content_hash_id = (
        SELECT content_hash_id
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        GROUP BY content_hash_id
        HAVING COUNT(*) >= 120   -- enough days to have a real 90+30 window
        LIMIT 1
    )
    ORDER BY report_date
""").df()

sample.head(15)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,sessions_organic
0,2025-02-04,client_9958f0a7ae1df715,content_199a7503ef4984b6,3,0,0
1,2025-02-06,client_9958f0a7ae1df715,content_199a7503ef4984b6,4,0,0
2,2025-02-07,client_9958f0a7ae1df715,content_199a7503ef4984b6,5,0,0
3,2025-02-08,client_9958f0a7ae1df715,content_199a7503ef4984b6,8,1,0
4,2025-02-09,client_9958f0a7ae1df715,content_199a7503ef4984b6,5,0,0
5,2025-02-10,client_9958f0a7ae1df715,content_199a7503ef4984b6,9,0,0
6,2025-02-11,client_9958f0a7ae1df715,content_199a7503ef4984b6,6,0,0
7,2025-02-12,client_9958f0a7ae1df715,content_199a7503ef4984b6,12,0,0
8,2025-02-13,client_9958f0a7ae1df715,content_199a7503ef4984b6,2,0,0
9,2025-02-14,client_9958f0a7ae1df715,content_199a7503ef4984b6,6,0,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule assumes clean, evenly-spaced, sufficiently-dense data — an assumption my own sample already violates: report dates have gaps (e.g., February 5th missing from a page's history), and click volume for many pages is low enough that a single click swings the percentage change dramatically. A rule like "flag a 25% click decline over the trailing 90 days" either means a different date range per page (since "90 rows" isn't "90 calendar days" when rows are missing), or gets dominated by noise on low-volume pages if the threshold isn't adjusted per page. A model can learn these adjustments jointly — weighting a page's history density, its baseline volume, and its decline pattern together — instead of requiring me to hand-write a separate correction for every edge case I happen to notice in the data. That's the real argument for ML here: not that the concept of decline is complex, but that the data quality conditions under which decline must be detected are messy enough that a single static rule breaks in predictable, evidence-backed ways.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.